# 2 · Build a dataset

Start with ten labels from notebook 0. Find a Sentinel-2 image for each label's place and day, then save the pairs:

```text
data/my_dataset/
├── train/  sentinel-2-l2a/<tile>.tif  label/<tile>.tif
├── val/    ...
├── test/   ...
└── catalog.parquet
```

We will also save six forest scenes for prediction in notebook 3. This notebook needs internet access.


In [ ]:
from pathlib import Path

from dask.diagnostics.progress import ProgressBar
from huggingface_hub import snapshot_download

import xarray as xr

import geosave as gs
from geosave.geodata import io

# Show progress when reading pixels.
pbar = ProgressBar(minimum=1)
pbar.register()

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300", repo_type="dataset", local_dir="data/dynamic_world"
    )
)

## Read a label's place and date

Its `anchor` describes geometry and time for searching. Its `geobox` defines the exact pixel grid for loading.


In [ ]:
label_path = sorted((root / "train/labels").glob("*.tif"))[0]
label = gs.open_raster(label_path)

anchor = label.gs.anchor
grid = label.gs.geobox
place = anchor.format("{lat:.2f}, {lon:.2f} · {start:%Y-%m-%d} · {width:g} m across")
f"{place} at {abs(grid.resolution.x):g} m"

In [ ]:
label.gs.plot()

## Find the image

A [STAC](https://stacspec.org) catalog lists scenes by place and date.

Choose four Sentinel-2 bands. Request cloud-cover ordering and keep one scene. Service ordering can vary; inspect the returned cloud cover when choosing production imagery. Search the label's own day so the image matches the label.


In [ ]:
bands = ["B02", "B03", "B04", "B08"]

client = gs.StacClient.open("planetary_computer")
collection = client.collection("sentinel-2-l2a")
query = gs.StacQuery(collections=["sentinel-2-l2a"], max_items=1).sort_by(
    "properties.eo:cloud_cover"
)
options: io.StacLoadOptions = dict(bands=bands, dtype="uint16", nodata=0)

Search the full day without loading pixels.


In [ ]:
day = anchor.replace(timespan=anchor.format("{start:%Y-%m-%d}"))
selected = client.search_items(query.set_anchor(day))
[(item.id, item.properties.get("eo:cloud_cover")) for item in selected]

Load the image on the label's grid. Record its reflectance scale and RGB bands. Plotting reads the pixels.


In [ ]:
image = io.load_dataset(
    selected, geobox=grid, collection=collection, patch_url=client.patch_url, **options
)
image = image.gs.write_packing(1e-4)
image = image.gs.write_rgb(red="B04", green="B03", blue="B02")

pair = xr.merge([image.gs.to_datatree("sentinel-2-l2a"), label.gs.to_datatree("label")])
pair.gs.plot(cols=2)

## Save and catalog the pair

`to_catalog` writes each group as a Cloud-Optimized GeoTIFF and returns a catalog of what it wrote: a GeoDataFrame with one STAC Item per group and date. `{collection}` gives each group its own folder; both use the label's filename.

A catalog stores file paths and metadata, not pixels. Add `split` as an ordinary column.

In [ ]:
folder = Path("data/my_dataset")

items = pair.gs.to_catalog(
    folder / "train",
    path_format=f"{{collection}}/{label_path.stem}.tif",
    id_format=label_path.stem,
    overwrite=True,
)
items["split"] = "train"
items[["id", "collection", "datetime", "split"]]

## Build all three splits

`fetch_image` loads an image. `save_tile` saves and catalogs one image/label pair. `build_dataset` repeats it and returns the catalog of one split.

In [ ]:
from tqdm.auto import tqdm


def fetch_image(anchor, grid):
    selected = client.search_items(query.set_anchor(anchor))
    image = io.load_dataset(
        selected,
        geobox=grid,
        collection=collection,
        patch_url=client.patch_url,
        **options,
    )
    image = image.gs.write_packing(1e-4)
    return image.gs.write_rgb(red="B04", green="B03", blue="B02")


def save_tile(label_path, folder):
    label = gs.open_raster(label_path)
    anchor = label.gs.anchor
    day = anchor.replace(timespan=anchor.format("{start:%Y-%m-%d}"))
    image = fetch_image(day, label.gs.geobox)
    pair = xr.merge(
        [image.gs.to_datatree("sentinel-2-l2a"), label.gs.to_datatree("label")]
    )
    return pair.gs.to_catalog(
        folder,
        path_format=f"{{collection}}/{label_path.stem}.tif",
        id_format=label_path.stem,
        overwrite=True,
    )


def build_dataset(labels, folder, split):
    catalogs = []
    for path in tqdm(labels, desc=split):
        catalogs.append(save_tile(path, folder / split))
    catalog = gs.GeoCatalog.concat(catalogs)
    catalog["split"] = split
    return catalog

Use six training labels, two validation labels and two test labels for this short example.


In [ ]:
pbar.unregister()

train_labels = sorted((root / "train/labels").glob("*.tif"))[:6]
val_labels = sorted((root / "val/labels").glob("*.tif"))[:2]
test_labels = sorted((root / "test/labels").glob("*.tif"))[:2]

train = build_dataset(train_labels, folder, "train")
val = build_dataset(val_labels, folder, "val")
test = build_dataset(test_labels, folder, "test")

catalog = gs.GeoCatalog.concat([train, val, test])
catalog.gs.to_geoparquet(folder / "catalog.parquet", overwrite=True)
catalog

Open the catalog on a map. Select rows, then use **Open selected items** to view their rasters.

The server runs in the background, so you can keep working in the next cells. Running this cell again replaces the explorer on the same port.


In [ ]:
explorer = catalog.gs.explore()
server = explorer.serve()

Stop the server when you are done looking.


In [ ]:
explorer.stop()

## Reopen the catalog

The saved GeoParquet works as a STAC source, searched like Planetary Computer.

In [ ]:
saved = gs.StacClient.open(folder / "catalog.parquet")
saved.search(gs.StacQuery(collections=["label"]))

## Pair images with labels

`sample` groups catalog rows into samples without opening a file. `pair` joins each Sentinel-2 Item with the label of the same place within `tolerance`. A label is dated at midnight and an image at its moment of capture, so allow a day.

In [ ]:
samples = catalog.gs.sample(pair=["sentinel-2-l2a", "label"], tolerance="1D")
len(samples), samples[0][["id", "collection", "split"]]

Each sample opens as a lazy DataTree with collection parents and grid leaves, `/{collection}/{res}`. Notebook 3 reads those leaves and cuts the trees into model-sized chips.


In [ ]:
samples[0].gs.to_datatree()

## Start from a shape

Open the forest outline in `khdtk.geojson` and give it a 10 m grid. Fetch one scene from each month for prediction.


In [ ]:
from odc.geo.geobox import GeoBox

region = gs.open_vector("khdtk.geojson")
region_anchor = gs.GeoAnchor.from_geometry(region)
# KHDTK is in UTM zone 49S. Reuse this grid for every month's image.
region_grid = GeoBox.from_geopolygon(
    region_anchor.geometry, crs="EPSG:32749", resolution=10
)

months = ["2019-05", "2019-06", "2019-07", "2019-08", "2019-09", "2019-10"]
pictures = []
for month in months:
    picture = fetch_image(region_anchor.replace(timespan=month), region_grid)
    pictures.append(picture)

season = gs.transform.temporal.concat(pictures).compute()
season

## Add NDVI

NDVI uses the red and near-infrared bands to measure greenness. Add it with xarray's `assign`. Keep the original four-band `season` for the model.


In [ ]:
ndvi = gs.features.ndvi(season, nir="B08", red="B04")
with_ndvi = season.assign(ndvi=ndvi)
with_ndvi

In [ ]:
with_ndvi.gs.explore("ndvi", basemap="Imagery", cmap="RdYlGn") * region

Save and catalog the six scenes for notebook 3. Each file keeps its acquisition date; these are individual scenes, not monthly composites.

In [ ]:
khdtk = Path("data/khdtk")

season_catalog = season.gs.to_catalog(
    khdtk,
    path_format="{collection}/{time:%Y%m%d}.tif",
    collection="sentinel-2-l2a",
    id_format="{start:%Y%m%d}",
    overwrite=True,
)
season_catalog.gs.to_geoparquet(khdtk / "catalog.parquet", overwrite=True)
season_catalog[["id", "collection", "datetime"]]

## What happened

We saved image/label pairs in fixed splits and six forest scenes for prediction. Notebook 3 uses their catalogs to make model inputs.


## Your turn

Easy to advanced; solutions are folded under each exercise.

### 1 · Another year over the forest (easy)

`fetch_image` takes an anchor and a grid. Fetch the forest in July 2020 and plot it beside July 2019. What changed?

<details><summary>Solution</summary>

```python
july = {
    "2019-07": fetch_image(region_anchor.replace(timespan="2019-07"), region_grid),
    "2020-07": fetch_image(region_anchor.replace(timespan="2020-07"), region_grid),
}
xr.merge([image.gs.to_datatree(month) for month, image in july.items()]).gs.plot(cols=2)
```
</details>

In [ ]:
# Your code here

### 2 · More pictures to choose from (easy)

Make a second query that keeps up to five pictures under 30% cloud, and search the first label's day. How many did it find, and how cloudy are they?

<details><summary>Solution</summary>

```python
loose = gs.StacQuery(collections=["sentinel-2-l2a"], max_items=5)
loose = loose.set_filter("eo:cloud_cover <= 30").sort_by("properties.eo:cloud_cover")
client.search(loose.set_anchor(day))
```

The new query leaves `query` unchanged. Searching reads no pixels; load options belong to `io.load_dataset`.
</details>


In [ ]:
# Your code here

### 3 · Earlier or later scenes (medium)

Dynamic World uses the label's own day. For another project, you may want a wider search window. Compare the matches below with the same-day search; do not silently substitute these observations for the labelled scene.

<details><summary>Solution</summary>

```python
from datetime import timedelta

start, end = day.timespan
window = day.replace(
    timespan=(
        start - timedelta(days=1),  # -1D: include the previous day
        end + timedelta(days=1),    # +1D: include the next day
    )
)
client.search(query.set_anchor(window))
```

This changes which acquisitions are eligible. It does not change the label's validity or average raster pixels.
</details>

In [ ]:
# Your code here

### 4 · When the archive has nothing (medium)

Some places and days have no archived scene. Find out what `fetch_image` does then, and change `build_dataset` so it skips those labels and returns the ones it skipped. Run it on 30 training labels.

<details><summary>Hint</summary>

`io.load_dataset` raises `AnchorFetchError`, from `geosave.geodata.errors`, when the search matches nothing. Catch it around `save_tile` and keep the label's path in a list.
</details>

In [ ]:
# Your code here

### 5 · Another layer (advanced)

Add elevation to every pair: Planetary Computer's `cop-dem-glo-30` collection. Plot picture, label and elevation together, and save them as `save_tile` does, with a path for the elevation too.

<details><summary>Hint</summary>

Make a query with `gs.StacQuery(collections=["cop-dem-glo-30"])` and apply `set_anchor(anchor.replace(timespan=None))`: elevation is not tied to the label's day. Search its Items and load them on `label.gs.geobox`, passing the collection metadata and `client.patch_url`. Its band is called `data`; `rename({"data": "elevation"})` gives it a better name.
</details>

In [ ]:
# Your code here

### 6 · Your own place (project)

Draw a field, a park or a forest you know at [geojson.io](https://geojson.io), save it next to this notebook, and build its season for another year: fetch, explore its NDVI, save and catalog it, as we did for the forest. Does NDVI show the season you expect?

In [ ]:
# Your code here